<a href="https://colab.research.google.com/github/NirtonAfonso/tech-challenge-fase3-medflow-ai/blob/develop/notebooks/03_rag_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir no Colab"/></a>

# 03 — Pipeline de RAG: construção e avaliação

**Tech Challenge Fase 3 — MedFlow AI**

| | |
|---|---|
| Runtime | **CPU** |
| Como rodar | `Ambiente de execução` → `Executar tudo` |
| Saída no Drive | `MedFlowAI_Fase3/03_rag/` |

Cada seção responde a uma **pergunta de pesquisa** explícita, seguida de método, resultado,
interpretação, trade-off e limitação — a estrutura exigida pelo feedback da Fase 1.

In [ ]:
# @title ▶ Bootstrap — execute esta célula primeiro (Colab ou local)
#
# Prepara tudo do zero em um runtime Colab novo: monta o Google Drive, clona a
# branch `develop`, instala as dependências e cria a estrutura de saída.
# Rodando localmente, detecta o repositório e pula clone/Drive.

import os
import pathlib
import subprocess
import sys

REPO_URL = "https://github.com/NirtonAfonso/tech-challenge-fase3-medflow-ai.git"
REPO_BRANCH = "develop"
REPO_DIR = "tech-challenge-fase3-medflow-ai"
NOTEBOOK_ID = "03_rag"
REQUIREMENTS = "requirements-colab.txt"

IN_COLAB = "google.colab" in sys.modules or bool(os.environ.get("COLAB_RELEASE_TAG"))


def _run(*args, **kwargs):
    return subprocess.run(list(args), check=kwargs.pop("check", True), **kwargs)


def _pip(*args):
    _run(sys.executable, "-m", "pip", *args)


def _tem_torch_cuda() -> bool:
    try:
        import torch

        return torch.cuda.is_available()
    except Exception:
        return False


# --- 1. Google Drive ---------------------------------------------------------
if IN_COLAB:
    try:
        from google.colab import drive

        drive.mount("/content/drive")
        print("Google Drive montado em /content/drive")
    except Exception as erro:
        print(f"ATENÇÃO: falha ao montar o Drive ({erro}).")
        print("Os resultados ficarão apenas em /content e serão PERDIDOS ao encerrar a sessão.")

# --- 2. Repositório ----------------------------------------------------------
def _raiz_local() -> pathlib.Path | None:
    atual = pathlib.Path.cwd()
    for candidato in [atual, *atual.parents]:
        if (candidato / "src" / "medflow_ai").exists():
            return candidato
    return None


raiz = _raiz_local()
if raiz is None:
    destino = pathlib.Path("/content" if IN_COLAB else ".") / REPO_DIR
    if destino.exists():
        _run("git", "-C", str(destino), "fetch", "--depth", "1", "origin", REPO_BRANCH)
        _run("git", "-C", str(destino), "checkout", REPO_BRANCH)
        _run("git", "-C", str(destino), "pull", "--ff-only", "origin", REPO_BRANCH)
    else:
        # Sempre com --branch explícita: nunca clonar a default implicitamente.
        _run("git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(destino))
    raiz = destino.resolve()

os.chdir(raiz)
if str(raiz / "src") not in sys.path:
    sys.path.insert(0, str(raiz / "src"))
print(f"Raiz do projeto: {raiz}")

# --- 3. Dependências ---------------------------------------------------------
_pip("install", "-q", "-U", "pip")
_pip("install", "-q", "-r", REQUIREMENTS)
_pip("install", "-q", "-e", ".")

# --- 4. Diagnóstico ----------------------------------------------------------
import platform

from medflow_ai.colab import ensure_structure, git_info, in_colab, write_run_metadata

_git = git_info(raiz)
print("\n" + "=" * 78)
print(f"Python        : {platform.python_version()}")
print(f"Ambiente      : {'Google Colab' if IN_COLAB else 'local'}")
print(f"Branch        : {_git['branch']}")
print(f"Commit        : {_git['commit']}")
print("=" * 78)

# --- 6. Estrutura de saída (Drive no Colab, artifacts/colab localmente) -------
PASTAS = ensure_structure(NOTEBOOK_ID)
print("\nEstrutura de saída:")
for _nome, _caminho in sorted(PASTAS.items()):
    print(f"  {_nome:22s} {_caminho}")

RUN_META = write_run_metadata(NOTEBOOK_ID)
print(f"\nMetadados da execução: {RUN_META}")


## 1. Como o corpus vira chunks recuperáveis?

**Método.** Document Loaders do LangChain → uma unidade por seção `##` →
`RecursiveCharacterTextSplitter` → `chunk_id` estável por conteúdo.

In [ ]:
from medflow_ai.rag.chunking import chunk_documents
from medflow_ai.rag.loaders import load_protocol_documents

secoes = load_protocol_documents()
chunks = chunk_documents(secoes, chunk_size=400, chunk_overlap=100)
print(f"{len(secoes)} seções → {len(chunks)} chunks")

print("\nMetadados de rastreabilidade de um chunk:")
for chave, valor in chunks[0].metadata.items():
    print(f"  {chave:16s}: {valor}")

**Interpretação.** Todo chunk carrega `doc_id`, `section_id`, `chunk_id`, versão e vigência do
documento. Sem isso, "citar a fonte" seria apenas escrever um nome de arquivo no fim da resposta.

## 2. Qual backend de embedding usar?

**Pergunta.** É possível medir RAG de forma reprodutível sem depender de download de modelo?

**Método.** O backend padrão é um *hashing trick* determinístico (n-gramas de palavra e de caractere,
TF sublinear, normalização L2). Um backend denso (`sentence-transformers`) fica disponível como opção,
testado na seção 7.

In [ ]:
import numpy as np

from medflow_ai.rag.embeddings import get_embeddings

emb = get_embeddings("hashing")
print("backend:", emb.name)

consulta = "quando repetir o TSH após ajustar a dose de levotiroxina"
textos = [
    "O controle do TSH deve ser feito 6 a 8 semanas após início ou ajuste de dose.",
    "A radiografia de tórax não é rotina na crise asmática.",
    "Coletar hemoculturas antes do antimicrobiano na sepse.",
]
q = np.array(emb.embed_query(consulta))
for texto, vetor in zip(textos, emb.embed_documents(textos)):
    print(f"  cos={float(np.array(vetor) @ q):+.3f}  {texto[:70]}")

**Trade-off registrado.** O backend por hashing é essencialmente lexical: não captura sinonímia
profunda. Em compensação é determinístico, roda em CI sem GPU e permite que qualquer pessoa regenere as
métricas do relatório com um comando. A avaliação da seção 4 mede o custo dessa escolha.

## 3. Como o índice é construído e persistido?

In [ ]:
from medflow_ai.rag.vector_store import MedFlowVectorStore

store = MedFlowVectorStore.from_documents(chunks, emb)
print(f"{len(store)} chunks indexados")

for doc, score in store.similarity_search_with_score("valores críticos comunicados pelo laboratório", k=3):
    print(f"  {score:.3f}  {doc.metadata['citation'][:88]}")

caminho_indice = store.save(PASTAS["indexes"] / "hashing_chunk400", extra={"chunk_size": 400, "chunk_overlap": 100})
print(f"\nÍndice salvo em {caminho_indice}")

## 4. Qual configuração de recuperação é a melhor? (experimento principal)

**Pergunta de pesquisa.** Dadas 40 perguntas clínicas com seção-ouro conhecida, qual combinação de
estratégia × `k` × tamanho de chunk recupera o trecho certo com mais frequência e em posição mais alta?

**Método.** Grade completa de 4 estratégias × 3 valores de `k` × 3 tamanhos de chunk = 36 configurações.
Métricas: `hit@k` (seção correta entre os k primeiros), `doc_hit@k` (documento correto) e `MRR`.

In [ ]:
from medflow_ai.evaluation.rag_eval import load_benchmark, run_experiment_grid, save_results

benchmark = load_benchmark()
print(f"{len(benchmark)} perguntas com gabarito")
print("Exemplo:", benchmark[0].question, "→", benchmark[0].gold_section_id)

resultados = run_experiment_grid()
caminhos_rag = save_results(resultados, PASTAS["artifacts"])
print(f"\n{len(resultados)} configurações avaliadas · artefatos em {caminhos_rag['csv']}")

In [ ]:
import pandas as pd

tabela = pd.DataFrame([{k: v for k, v in r.to_dict().items() if k != "falhas"} for r in resultados])
tabela.sort_values(["hit_at_k", "mrr"], ascending=False).head(10)

In [ ]:
import matplotlib
matplotlib.use("Agg") if not IN_COLAB else None
import matplotlib.pyplot as plt

pivo = tabela[tabela.chunk_size == 400].pivot(index="k", columns="strategy", values="hit_at_k")
eixo = pivo.plot(marker="o", figsize=(8, 4.5))
eixo.set_title("Pergunta: qual estratégia recupera a seção correta com mais frequência? (chunk=400)")
eixo.set_xlabel("k (trechos recuperados)"); eixo.set_ylabel("hit@k")
eixo.set_xticks(sorted(tabela.k.unique())); eixo.grid(alpha=.3)
plt.tight_layout()
grafico_hit = PASTAS["artifacts"] / "rag_hit_at_k.png"
plt.savefig(grafico_hit, dpi=150); plt.show()
print("gráfico salvo em", grafico_hit)

In [ ]:
eixo = tabela.pivot_table(index="chunk_size", columns="strategy", values="mrr").plot(
    kind="bar", figsize=(8, 4.5))
eixo.set_title("Pergunta: o tamanho do chunk muda a posição do acerto? (MRR médio)")
eixo.set_ylabel("MRR"); eixo.grid(alpha=.3, axis="y")
plt.tight_layout()
grafico_mrr = PASTAS["artifacts"] / "rag_mrr_por_chunk.png"
plt.savefig(grafico_mrr, dpi=150); plt.show()
print("gráfico salvo em", grafico_mrr)

**Resultado (regenerado a cada execução, em `03_rag/artifacts/rag_experiments.csv`).**

- `hybrid`, `k=5`, `chunk=400` obteve o melhor `hit@5`;
- `doc_hit@5` fica próximo de 1,0 em quase todas as configurações — o sistema quase sempre acerta o
  **documento**, e erra com mais frequência a **seção** dentro dele;
- `mmr` degrada nitidamente com chunks maiores.

**Interpretação.** A penalização de redundância do MMR é contraproducente neste corpus: várias seções do
mesmo protocolo são legitimamente relevantes para a mesma pergunta, e o MMR as afasta em favor de
diversidade artificial. Por isso a configuração de produção adotou `hybrid`.

**Trade-off.** `hybrid` custa cerca de 3–4× a latência da busca densa pura (fusão de dois rankings). Na
escala deste corpus isso é irrelevante (sub-milissegundo); em um corpus hospitalar real, mereceria nova
medição.

**Limitação declarada.** O benchmark é construído sobre o **mesmo corpus** indexado. Ele mede qualidade
de recuperação, **não** generalização para documentos inéditos. As perguntas foram escritas com
vocabulário diferente do texto-fonte para reduzir casamento lexical trivial, mas isso não elimina a
limitação.

## 5. Onde o RAG erra?

In [ ]:
from medflow_ai.evaluation.rag_eval import evaluate_retriever
from medflow_ai.rag.retriever import ProtocolRetriever

recuperador = ProtocolRetriever(store, strategy="hybrid", k=5)
metricas = evaluate_retriever(recuperador, benchmark, k=5)
print(f"hit@5={metricas.hit_at_k} doc_hit@5={metricas.doc_hit_at_k} MRR={metricas.mrr}")
print("Perguntas com falha:", metricas.falhas)

por_id = {item.id: item for item in benchmark}
for falha in metricas.falhas[:4]:
    item = por_id[falha]
    print(f"\n[{item.id}] {item.question}")
    print(f"  esperado : {item.gold_section_id}")
    for chunk in recuperador.retrieve(item.question, k=3):
        print(f"  obtido   : {chunk.section_id:20s} score={chunk.score:.3f}")

**Interpretação dos erros.** As falhas concentram-se em perguntas cujo vocabulário se afasta do
texto-fonte (paráfrase forte) — exatamente o ponto fraco previsto para um embedding lexical. A seção 7
testa se um embedding denso corrige parte disso.

## 6. O RAG melhora a resposta final? (ablação)

In [ ]:
from medflow_ai.fine_tuning.evaluate import compare_systems
from medflow_ai.llm.prompts import build_messages, format_protocol_block
from medflow_ai.llm.providers import get_chat_model

modelo = get_chat_model("template")

def com_rag(pergunta):
    mensagens = build_messages(question=pergunta, patient_context="",
                               protocol_context=format_protocol_block(recuperador.retrieve(pergunta)),
                               safety_status="SAFE")
    return str(modelo.invoke(mensagens).content)

def sem_rag(pergunta):
    mensagens = build_messages(question=pergunta, patient_context="",
                               protocol_context="Nenhum trecho de protocolo foi recuperado.",
                               safety_status="SAFE")
    return str(modelo.invoke(mensagens).content)

comparacao = compare_systems({"sem_rag": sem_rag, "com_rag": com_rag}, output_dir=PASTAS["artifacts"])
pd.DataFrame([{k: v for k, v in r.to_dict().items() if k != "exemplos"} for r in comparacao])

**Interpretação.** Sem RAG, o gerador não tem o que citar: `citacao_presente` e `groundedness` caem
a zero e o sistema corretamente declara falta de evidência em vez de inventar. Com RAG, todas as
respostas citam fonte e a maioria cita o documento correto.

**Limitação.** O gerador desta ablação é o baseline extrativo determinístico, **não** uma LLM. A mesma
medição com o modelo base e o fine-tuned está no notebook 02.

## 7. (Opcional) Embedding denso no Colab

**Pergunta.** Um embedding denso multilíngue corrige as falhas de paráfrase da seção 5?

Esta seção baixa `sentence-transformers` (~100 MB na primeira vez). Ela é **aditiva**: os resultados do
backend `hashing` acima continuam sendo os números oficiais do relatório, e os daqui são reportados
separadamente. Ponha `EXECUTAR_DENSO = False` para pular.

In [ ]:
EXECUTAR_DENSO = True  # ponha False para pular esta seção

resultado_denso = None
if EXECUTAR_DENSO:
    try:
        import sentence_transformers  # noqa: F401
    except ImportError:
        print("Instalando sentence-transformers (uma vez por runtime)...")
        import subprocess, sys
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "sentence-transformers"], check=True)

    try:
        emb_denso = get_embeddings("sentence_transformers")
        store_denso = MedFlowVectorStore.from_documents(chunks, emb_denso)
        for estrategia in ("dense", "mmr", "hybrid"):
            metrica = evaluate_retriever(
                ProtocolRetriever(store_denso, strategy=estrategia, k=5), benchmark, k=5,
                chunk_size=400, chunk_overlap=100, embedding_name=emb_denso.name,
            )
            print(f"{estrategia:7s} hit@5={metrica.hit_at_k:.3f} doc_hit@5={metrica.doc_hit_at_k:.3f} MRR={metrica.mrr:.3f}")
            if estrategia == "hybrid":
                resultado_denso = metrica
    except Exception as erro:
        print(f"Seção densa não executada: {erro}")
        print("Os números oficiais do relatório continuam sendo os do backend 'hashing'.")
else:
    print("Seção densa pulada por configuração.")

In [ ]:
import json

if resultado_denso is not None:
    caminho_denso = PASTAS["artifacts"] / "rag_dense_embeddings.json"
    caminho_denso.write_text(json.dumps({
        "aviso": ("Resultado ADITIVO do backend denso. Não substitui as métricas oficiais, "
                  "medidas com o backend determinístico 'hashing'."),
        "melhor_hybrid_k5_chunk400": resultado_denso.to_dict(),
    }, ensure_ascii=False, indent=2), encoding="utf-8")
    print("salvo em", caminho_denso)
else:
    caminho_denso = None
    print("Nenhum resultado denso a salvar nesta execução.")

## 8. Persistência no Google Drive

In [ ]:
# Persistência no Google Drive — uma execução só termina quando os resultados
# saem de /content. Fora do Colab, os mesmos arquivos vão para artifacts/colab/.
from medflow_ai.colab import persist, summarize

_relatorios = [
    persist(
        [caminhos_rag["json"], caminhos_rag["csv"], grafico_hit, grafico_mrr,
         PASTAS["artifacts"] / "generation_comparison.json"]
        + ([caminho_denso] if caminho_denso else []),
        PASTAS["artifacts"],
    ),
    persist([caminho_indice], PASTAS["indexes"]),
]

print(summarize(_relatorios, titulo="RESUMO DA PERSISTÊNCIA — 03_rag"))
